# Agentic Portfolio Construction — Full Pipeline Demo

Human-capital-aware portfolio construction. A client's total wealth is
**Financial Capital + Human Capital** (PV of future earnings), so the portfolio's
equity budget is set net of the equity risk the client already carries through
their career.

```
Profile ──┐
          ├─► Allocation ◄──► Risk        Loop A  (FLAG,  ≤3)
Research ─┘        │
                   ▼
              Compliance                  Loop B  (FAIL,  ≤2)
                   │
                   ▼
             AdvisorPackage
```

| § | Contents |
|---|---|
| 1 | Environment setup |
| 2 | Data fetch |
| 3 | Profile Agent |
| 4 | Research Agent |
| 5 | Both feedback loops (`run_pipeline`) |
| 6 | Full `AdvisorPackage` |

Every quantitative decision is deterministic. The LLM proposes the risky-sleeve
composition and writes narrative; `agents/allocation/validator.py` re-checks it
against the same hard limits the optimizer enforces. Without an
`ANTHROPIC_API_KEY` the Allocation Agent falls back to the Black-Litterman solve,
so the whole notebook runs offline against the parquet cache.

## 1 · Environment setup

Puts the repo root on `sys.path`, loads `.env`, routes every agent's `INFO` logs
into the notebook (the feedback loops in §5 are only legible with logging on), and
defines the display helpers the later sections use.

In [ ]:
import os, sys, json, logging
from pathlib import Path

PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

try:
    from dotenv import load_dotenv
    load_dotenv(Path(PROJECT_ROOT) / ".env")
    print(".env loaded")
except ImportError:
    print("dotenv not installed — run: pip install python-dotenv")

FRED_KEY      = os.environ.get("FRED_API_KEY")
ANTHROPIC_KEY = os.environ.get("ANTHROPIC_API_KEY")

print(f"FRED_API_KEY set:      {bool(FRED_KEY)}")
print(f"ANTHROPIC_API_KEY set: {bool(ANTHROPIC_KEY)}"
      f"{'' if ANTHROPIC_KEY else '   -> Allocation falls back to the deterministic BL solve'}")

# Agent logs in the notebook. force=True overrides config set by imported modules.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(name)-34s | %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
    force=True,
)
log = logging.getLogger("demo")

RULE = "=" * 88


def banner(text):
    print("\n" + "#" * 88 + "\n#  " + text + "\n" + "#" * 88)


def section(title):
    print("\n" + RULE + "\n" + title + "\n" + RULE)


def show(title, model):
    """Print a Pydantic model (or dict) as indented JSON under a titled rule."""
    section(title)
    if hasattr(model, "model_dump_json"):
        print(model.model_dump_json(indent=2))
    else:
        print(json.dumps(model, indent=2, default=str))


print("\nsetup complete — Python", sys.version.split()[0])

## 2 · Data fetch

The pipeline reads the parquet cache in `data/storage/`. Two things are checked:
the required files exist, **and** the cached CRSP data actually covers the current
asset universe.

That second check matters. The universe is now **33 single stocks (3 per GICS
sector) + 5 defensive funds**, replacing the old 24-ETF set where `SPY` held the
`XL*` sector ETFs' constituents and `AGG` held the treasury funds' — so a 10%
single-name cap on `SPY` and 10% on `XLK` never actually bounded technology at
20%. An older ETF cache has all the *files* but none of the *stock* PERMNOs, which
would fail deep inside the optimizer rather than here.

In [ ]:
from agents.allocation.adapters import DEFAULT_TICKERS, ETF_SECTORS

REQUIRED = [
    "crsp_monthly.parquet", "crsp_daily.parquet", "ff_risk_factors.parquet",
    "fred_macro.parquet",   "fred_dgs10.parquet",  "bls_oes.parquet",
    "ff12_monthly.parquet", "permno_map.json",     "mkt_cap_weights.json",
]
storage = Path("data/storage")

missing_files = [f for f in REQUIRED if not (storage / f).exists()]
for f in REQUIRED:
    print(f"  {'OK     ' if (storage / f).exists() else 'MISSING'}  {f}")

permno_path = storage / "permno_map.json"
covered     = json.loads(permno_path.read_text()) if permno_path.exists() else {}
uncovered   = [t for t in DEFAULT_TICKERS if t not in covered]
needs_fetch = bool(missing_files) or bool(uncovered)

# Universe composition, by sector.
by_sector = {}
for t in DEFAULT_TICKERS:
    by_sector.setdefault(ETF_SECTORS[t], []).append(t)

section(f"ASSET UNIVERSE — {len(DEFAULT_TICKERS)} instruments, {len(by_sector)} sectors")
for sec in sorted(by_sector):
    print(f"  {sec:24s} {' '.join(by_sector[sec])}")

print(f"\nIn CRSP cache: {len(DEFAULT_TICKERS) - len(uncovered)}/{len(DEFAULT_TICKERS)}")
if uncovered:
    print(f"  NOT cached ({len(uncovered)}): {' '.join(uncovered)}")
print("\nCache is complete and covers the universe — continue to §3."
      if not needs_fetch else
      "\nRun the next cell to fetch. Needs WRDS credentials (interactive prompt).")

In [ ]:
# Guarded fetch. Auto-runs only when §2 found the cache short; set True to force.
RUN_DATA_FETCH = needs_fetch

if RUN_DATA_FETCH:
    from data.fetch.wrds import (
        get_connection, fetch_crsp_monthly, fetch_crsp_daily, fetch_ff_factors,
    )

    # The existing cache is the only runnable state — keep a copy before overwriting.
    import shutil
    from datetime import datetime
    stamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup = storage / f"backup_{stamp}"
    backup.mkdir(exist_ok=True)
    for name in ("crsp_monthly.parquet", "crsp_daily.parquet",
                 "permno_map.json", "mkt_cap_weights.json"):
        if (storage / name).exists():
            shutil.copy2(storage / name, backup / name)
    print(f"Backed up existing cache -> {backup.name}/\n")

    conn = get_connection()                 # prompts for WRDS username + password
    print(f"Connected — fetching CRSP for {len(DEFAULT_TICKERS)} tickers")
    print("Watch for 'no CRSP PERMNO found' — a dropped ticker silently shrinks the universe.\n")

    # force=True: these short-circuit on an existing parquet, and one exists.
    fetch_crsp_monthly(DEFAULT_TICKERS, conn=conn, force=True)
    fetch_crsp_daily(DEFAULT_TICKERS, conn=conn, force=True)

    # Universe-independent series — only if absent.
    if not (storage / "ff_risk_factors.parquet").exists():
        fetch_ff_factors(conn=conn)
    if not (storage / "fred_macro.parquet").exists():
        from data.fetch.fred import fetch_fred_macro;  fetch_fred_macro(fred_api_key=FRED_KEY)
    if not (storage / "fred_dgs10.parquet").exists():
        from data.fetch.fred import fetch_fred_dgs10;  fetch_fred_dgs10(fred_api_key=FRED_KEY)
    if not (storage / "bls_oes.parquet").exists():
        from data.fetch.bls import fetch_bls_oes;      fetch_bls_oes()
    if not (storage / "ff12_monthly.parquet").exists():
        from data.fetch.factors import fetch_ff12;     fetch_ff12()

    print("\nCache populated for the current universe.")
else:
    print("Cache covers the universe — skipping fetch. Set RUN_DATA_FETCH = True to force.")

In [ ]:
# Estimation window actually available to Black-Litterman.
#
# build_returns_matrix() ends in pivot.dropna(), so the covariance sample is the
# INTERSECTION of every ticker's history — one late listing truncates all of them.
# T/N is what black_litterman() inverts against; below ~5 the sample covariance is
# poorly conditioned.
import pandas as pd

if not uncovered:
    pm  = json.loads((storage / "permno_map.json").read_text())
    inv = {pm[t]: t for t in DEFAULT_TICKERS if t in pm}
    df  = pd.read_parquet(storage / "crsp_monthly.parquet")
    df  = df[df["permno"].isin(inv)].copy()
    df["ticker"] = df["permno"].map(inv)

    wide   = df.pivot(index="date", columns="ticker", values="ret")
    common = wide.dropna()
    T, N   = common.shape

    section("ESTIMATION WINDOW")
    print(f"  full panel      : {wide.shape[0]} months x {wide.shape[1]} assets")
    print(f"  common (dropna) : {T} months x {N} assets"
          f"   {common.index.min().date()} -> {common.index.max().date()}")
    print(f"  T/N             : {T / N:.2f}"
          f"{'   <-- thin; covariance poorly conditioned' if T / N < 5 else ''}")

    first = wide.apply(lambda c: c.first_valid_index()).sort_values(ascending=False)
    print("\n  latest-starting names (these set the window):")
    for tk, when in first.head(5).items():
        print(f"     {tk:6s} {when.date() if when is not None else 'no data'}")
else:
    print("Universe not cached yet — run the fetch cell above first.")

## 3 · Profile Agent

Turns each BLS occupation persona into a validated `ProfileAgentOutput`.

The chain that matters:

```
HC = salary x [1 - (1+r)^-n] / r                      annuity PV, r = FRED DGS10
implicit_equity_exposure = (HC / total_wealth) x β    equity already held via career
portfolio_equity_target  = effective_risk_budget - implicit_equity_exposure
```

`portfolio_equity_target` is the whole thesis in one number: the residual equity
capacity left *after* accounting for the career. It can go negative — a career
that already carries more equity risk than the total risk budget allows.

In [ ]:
from agents.profile.profile_agent import run_profile_agent

banner("PROFILE AGENT")
profiles = run_profile_agent(save=False)          # 9 BLS occupations at median salary
log.info("Profile Agent produced %d personas", len(profiles))

tbl = pd.DataFrame([{
    "client_id":   p.client_id,
    "career":      p.career_type,
    "age":         p.age,
    "FC":          f"{p.financial_capital:,.0f}",
    "HC":          f"{p.human_capital_valuation:,.0f}",
    "HC/FC":       f"{p.human_capital_valuation / p.financial_capital:.1f}x",
    "hc_type":     p.human_capital_type.value,
    "beta":        round(p.income_equity_beta, 3),
    "implicit_eq": round(p.implicit_equity_exposure, 3),
    "eq_target":   round(p.portfolio_equity_target, 3),
} for p in profiles])

section("PERSONA DIFFERENTIATION — human capital drives the equity target")
print(tbl.to_string(index=False))
print("\nA negative eq_target means the career alone already exceeds the client's")
print("total risk budget, so the optimizer is pushed toward the safe sleeve.")

In [ ]:
# Carry one persona through the rest of the notebook.
# Financial Analyst: mixed human capital, positive equity target — the
# non-degenerate middle case. Swap the id to contrast against bond-like
# (bls_25-1042_p50, professor) or equity-like (bls_15-1252_p50, software dev).
PERSONA_ID = "bls_13-2051_p50"

profile = next(p for p in profiles if p.client_id == PERSONA_ID)

section(f"SELECTED PERSONA — {profile.client_id} ({profile.career_type})")
print(f"  age                       {profile.age}")
print(f"  financial capital         {profile.financial_capital:>14,.0f}")
print(f"  human capital (PV)        {profile.human_capital_valuation:>14,.0f}")
print(f"  total wealth              {profile.total_wealth:>14,.0f}   "
      f"({profile.human_capital_pct_of_total:.1f}% human capital)")
print(f"  income sigma              {profile.income_volatility_sigma:>14.3f}")
print(f"  income-equity correlation {profile.income_equity_correlation:>14.3f}")
print(f"  income-equity beta        {profile.income_equity_beta:>14.3f}   "
      f"-> {profile.human_capital_type.value}")
print(f"  implicit equity exposure  {profile.implicit_equity_exposure:>14.3f}")
print(f"  effective risk budget     {profile.effective_risk_budget:>14.3f}")
print(f"  PORTFOLIO EQUITY TARGET   {profile.portfolio_equity_target:>14.3f}")
print(f"  employer sector           {profile.industry_exposure_sector}")
print(f"  risk tolerance            {profile.risk_tolerance_level.value}")
print(f"  llm_role                  {profile.llm_role.value}")

show(f"ProfileAgentOutput — {profile.client_id}", profile)

## 4 · Research Agent

Classifies the macro regime from 13 FRED series:
**PELT** structural breaks → **K-means** segment sanity check → **XGBoost**
month-by-month labelling → 6-month majority-vote smoothing.

Returns a `MacroRegimeSnapshot`. Note the snapshot exposes only
`regime_label`, `regime_confidence` and `regime_volatility` — the six raw FRED
signals (`yield_curve`, `fed_funds`, `cpi`, …) are **deprecated and no longer
populated**; reading them returns `None`. The full signal matrix stays in
`data/outputs/fred_macro_regimes.csv`.

In [ ]:
from agents.research.research_agent import run_research_agent

banner("RESEARCH AGENT")
macro = run_research_agent(save=False, validate_crsp=False, compare_models=False)

section("MACRO REGIME SNAPSHOT")
print(f"  as of                {macro.as_of}")
print(f"  regime               {macro.regime_label}")
print(f"  prior regime         {macro.prior_regime}")
print(f"  regime shift date    {macro.regime_shift_date}")
print(f"  confidence           {macro.regime_confidence:.1%}"
      f"{'   LOW — flagged pre-flight' if macro.is_low_confidence else ''}")
print(f"  regime volatility    {macro.regime_volatility:.4f}   (6m rolling std of credit spread)")
print(f"  regime change        {macro.regime_change_detected}")
print(f"  evidence attached    {macro.regime_change_evidence is not None}")

print("\n  allocation-facing surface (for_allocation()):")
for k, v in macro.for_allocation().items():
    print(f"     {k:20s} {v}")

In [ ]:
# Per-regime equity tilts — how the regime reaches the portfolio.
#
# The tilt is a bounded (+/-15%) relative adjustment to the client's equity
# target, derived from each regime's realised volatility vs the full-sample
# baseline. The orchestrator applies it at step 2b, gated on evaluate_rebalance().
from agents.orchestrator.orchestrator_agent import _load_regime_stats
from agents.research.rebalance import evaluate_rebalance

regime_stats = _load_regime_stats()

if regime_stats:
    section("REGIME-CONDITIONAL EQUITY TILTS")
    stats_df = pd.DataFrame([{
        "regime":     s.regime,
        "months":     s.n_months,
        "vol_annual": f"{s.vol_annual:.2%}",
        "vol_ratio":  f"{s.vol_ratio:.3f}",
        "tilt":       f"{s.tilt:+.2%}",
        "capped":     s.tilt_capped,
    } for s in regime_stats.values()])
    print(stats_df.to_string(index=False))
else:
    print("No regime stats — data/storage/fred_macro_regimes.parquet missing.")
    print("Run run_research_agent(save=True) once to write it.")

# Rebalance verdict for this client under this snapshot.
if macro.regime_change_evidence is not None:
    ev = evaluate_rebalance(macro, profile, regime_stats)
    section(f"REBALANCE VERDICT — {ev.decision.value.upper()}")
    print(f"  {ev.prior_regime}  ->  {ev.current_regime}")
    print(f"  equity target {ev.current_equity_target:.4f} -> {ev.proposed_equity_target:.4f} "
          f"(delta {ev.equity_target_delta:+.4f}, threshold {ev.materiality_threshold})")
    print(f"  evidence gates passed: {ev.evidence.gates_passed or '-'}")
    print(f"  evidence gates failed: {ev.evidence.gates_failed or '-'}")
    print(f"\n  {ev.explanation}")
else:
    print("\nSnapshot carries no regime_change_evidence — tilt will be skipped.")

## 5 · Both feedback loops — `run_pipeline`

`run_pipeline` is the control plane. It makes no quantitative or LLM decision of
its own; it sequences the agents and routes their verdicts.

```
1  pre-flight        low-confidence regime warning
2  discount rate     FRED DGS10, 4.4% fallback
2b regime tilt       evaluate_rebalance -> equity target, only when JUSTIFIED
3  LOOP A            Allocation <-> Risk    on FLAG, feed constraints_violated
                                            back as flag_constraints   (max 3)
4  assemble          ComplianceInput
5  LOOP B            Compliance <-> Allocation   on FAIL, route
                                            agent_feedback["allocation_agent"]  (max 2)
6  assemble          AdvisorPackage
```

Both loops are **bounded and typed**: `RiskOutput` refuses to validate a `FLAG` at
iteration 3 (it must be `REJECT`), and a `FLAG` with no violated constraint is
unconstructible. Watch the `INFO` logs below to see each iteration.

In [ ]:
from agents.orchestrator.orchestrator_agent import run_pipeline

banner("ORCHESTRATED PIPELINE — run_pipeline()")
package = run_pipeline(profile, macro, fred_api_key=FRED_KEY)

m = package.metadata
log.info("PACKAGE ASSEMBLED — risk=%s compliance=%s (risk_revisions=%d, compliance_revisions=%d)",
         m.final_risk_decision.value, m.final_compliance_status.value,
         m.risk_revisions, m.compliance_revisions)

In [ ]:
section("LOOP RESOLUTION")

print(f"  Loop A — Allocation <-> Risk")
print(f"     final decision      {m.final_risk_decision.value}")
print(f"     FLAG revisions      {m.risk_revisions} of 3 max"
      f"{'   (passed first try)' if m.risk_revisions == 0 else ''}")

print(f"\n  Loop B — Compliance <-> Allocation")
print(f"     final status        {m.final_compliance_status.value}")
print(f"     clearance           {package.compliance.clearance}")
print(f"     revisions           {m.compliance_revisions} of 2 max"
      f"{'   (passed first try)' if m.compliance_revisions == 0 else ''}")

# agent_feedback is the routing table Loop B consumes.
if package.compliance.agent_feedback:
    print("\n  Compliance routing table (agent_feedback):")
    for agent, items in package.compliance.agent_feedback.items():
        print(f"     -> {agent}  ({len(items)} item(s))")
        for it in items:
            print(f"          {it.check}: {it.action_required[:74]}")
else:
    print("\n  No agent_feedback — nothing was routed back.")

if m.pipeline_warnings:
    print("\n  Pipeline warnings:")
    for w in m.pipeline_warnings:
        print(f"     - {w}")

## 6 · Full `AdvisorPackage`

The validated end-to-end output: `{profile, macro, allocation, risk, compliance,
metadata}`.

One thing to read carefully — `proposed_portfolio` is **sleeve-relative**. It sums
to 1.0 across the risky sleeve, while `risky_weight` reports how much of financial
wealth goes into that sleeve at all. The remainder (`safe_weight`) is an abstract
risk-free asset, not a ticker, so it does not appear in the holdings table.

In [ ]:
MIN_W = 0.001    # hide dust positions below 0.1%
pkg   = package

banner("FULL ADVISOR PACKAGE")

section("SUMMARY")
print(f"  Client       {pkg.profile.client_id} · {pkg.profile.career_type} · age {pkg.profile.age}")
print(f"  Human cap.   {pkg.profile.human_capital_type.value} "
      f"(beta {pkg.profile.income_equity_beta:.2f}, "
      f"{pkg.profile.human_capital_pct_of_total:.0f}% of total wealth)")
print(f"  Eq. target   {pkg.profile.portfolio_equity_target:.3f}")
print(f"  Regime       {pkg.macro.regime_label} ({pkg.macro.regime_confidence:.0%} confidence)")
print(f"  Risk         {pkg.metadata.final_risk_decision.value} "
      f"({pkg.metadata.risk_revisions} revision(s))")
print(f"  Compliance   {pkg.metadata.final_compliance_status.value} "
      f"({pkg.metadata.compliance_revisions} revision(s)) · clearance={pkg.compliance.clearance}")

# ── Holdings ──
port   = pkg.allocation.proposed_portfolio
funded = sorted(((t, w) for t, w in port.items() if w >= MIN_W), key=lambda kv: -kv[1])

section(f"PORTFOLIO — {len(funded)} funded of {len(port)} instruments (risky sleeve, sums to 1.0)")
print(pd.DataFrame([
    {"Ticker": t, "Sector": ETF_SECTORS.get(t, "-"), "Weight": f"{w:.2%}"}
    for t, w in funded
]).to_string(index=False))

sector_w = {}
for t, w in funded:
    sector_w[ETF_SECTORS.get(t, "-")] = sector_w.get(ETF_SECTORS.get(t, "-"), 0) + w
print("\n  By sector (20% cap, 10% on the employer's sector "
      f"— {pkg.profile.industry_exposure_sector}):")
for s, w in sorted(sector_w.items(), key=lambda kv: -kv[1]):
    mark = "  <-- employer sector" if s == pkg.profile.industry_exposure_sector else ""
    print(f"     {s:24s} {w:6.2%}{mark}")

In [ ]:
section("PER-POSITION RATIONALE")
# Check 2.6 requires these to be distinct — one narrative copied across every
# ticker passes 2.2 individually but violates Reg BI's per-recommendation duty.
for t, _ in funded:
    print(f"  • {pkg.allocation.allocation_rationale[t]}\n")

unique = len(set(pkg.allocation.allocation_rationale.values()))
print(f"  {unique} unique rationale(s) across {len(pkg.allocation.allocation_rationale)} "
      f"position(s) — Check 2.6 {'PASS' if unique > 1 else 'would FAIL'}")

In [ ]:
section("RISK — regime stress tests")
print(pd.DataFrame([
    {"Regime": r,
     "Benchmark DD": f"{ev.benchmark_drawdown:.1%}",
     "Floor":        f"{ev.drawdown_floor:.1%}",
     "Portfolio DD": f"{ev.portfolio_drawdown:.1%}",
     "Result":       "PASS" if ev.passed else "FAIL"}
    for r, ev in pkg.risk.regime_evaluation.items()
]).to_string(index=False))

pl, sl = pkg.risk.position_limits, pkg.risk.sector_limits
print(f"\n  Position limits   {sum(v.passed for v in pl.values())}/{len(pl)} within cap")
print(f"  Sector limits     {sum(v.passed for v in sl.values())}/{len(sl)} within HC-adjusted cap")
if pkg.risk.portfolio_volatility_annual is not None:
    print(f"  Annualised vol    {pkg.risk.portfolio_volatility_annual:.2%}   (drives Check 2.5)")
else:
    print("  Annualised vol    not reported — Check 2.5 skipped")
if pkg.risk.market_regime:
    print(f"  Market regime     {pkg.risk.market_regime.value} "
          f"(effective drawdown cap {pkg.risk.effective_drawdown_cap:.0%})")

section("RISK — methodology audit trail (Compliance Check 1.3 reads this)")
d = pkg.risk.derivation
print(f"  drawdown_method       {d.drawdown_method}")
print(f"  concentration_method  {d.concentration_method}")
print(f"  sector_method         {d.sector_method}")
print(f"  hc_type               {d.hc_type}")
print(f"  rsu_concentration     {d.rsu_concentration}")
print(f"  data_source           {d.data_source}")

if pkg.risk.violations:
    print("\n  Violations:")
    for v in pkg.risk.violations:
        print(f"     - {v}")

In [ ]:
comp = pkg.compliance
section(f"COMPLIANCE — {comp.compliance_status.value} "
        f"(overall severity {comp.overall_severity.value})")

jobs = [("Job 1 — Risk-output audit",   "check_1"),
        ("Job 2 — Fiduciary content",   "check_2"),
        ("Job 3 — Robo-adviser (SEC IM 2017-02)", "check_3")]

print(pd.DataFrame([
    {"Job": label,
     "Passed":     sum(c.startswith(pfx) for c in comp.passed_checks),
     "Violations": sum(v.check.startswith(pfx) for v in comp.violations),
     "Result": "PASS" if not any(v.check.startswith(pfx) for v in comp.violations) else "REVIEW"}
    for label, pfx in jobs
]).to_string(index=False))

if comp.violations:
    print("\n  Violations (severity ladder: any HIGH -> FAIL + no clearance):")
    for v in comp.violations:
        print(f"     [{v.severity.value:6s}] {v.check}  ->  {v.responsible_agent}")
        print(f"              {v.description}")
        print(f"              rule: {v.rule_reference}")
        print(f"              action: {v.action_required}\n")

print(f"\n  RECOMMENDATION: {comp.recommendation}")

In [ ]:
# Raw validated package — uncomment to dump everything.
section("RAW PACKAGE")
print(f"  type              {type(pkg).__name__}")
print(f"  sections          {list(pkg.model_dump().keys())}")
print(f"  serialised size   {len(pkg.model_dump_json()):,} chars")
print("\n  Full JSON:  print(package.model_dump_json(indent=2))")

# print(package.model_dump_json(indent=2))

## Notes

- **Asset universe.** 33 single stocks (3 per GICS sector) + 5 defensive funds
  (`TLT`, `SHY`, `TIP`, `LQD`, `GLD`). Single names carry no hidden overlap, so
  the single-name and sector caps bound what they claim to. `AGG` and `BIL` are
  deliberately excluded — `AGG` holds the same treasuries as the funds above it,
  and `BIL` (0.6% vol) duplicates the abstract safe sleeve that `1 - w_fin`
  already represents.
- **Merger-broken tickers.** `RTX`, `LIN` and `PLD` are top-three by market cap
  but get fresh CRSP PERMNOs at their mergers (2020, 2018, 2011). Since
  `build_returns_matrix()` calls `pivot.dropna()`, including one would truncate
  the covariance sample for *all* assets. They are replaced by `HON`, `APD`, `SPG`.
- **Regime tilt.** §4's verdict gates whether the regime moves the book. For the
  current BLS personas human capital is 5.5x–25.5x financial capital, so the
  equity target saturates the achievable 0–100% range and no tilt is material —
  `evaluate_rebalance` returns `CHURN` and the target is left untouched. That is
  the model's finding, not a failure; see `rebalance.corner_solution()`.
- **Known gap.** The intake→profile weld is still open: `ExtractedProfile.statements`
  are not folded into `ProfileAgentOutput.client_statements`, so Compliance Job 3.2
  passes vacuously in a live run.